# MedGemma term-level billing NER on MDACE — Colab Runner (T4 free tier)

Zero-shot evaluation of **google/medgemma-4b-it** on MDACE billing evidence:
the phrases human medical coders highlighted in a note to justify the billing
codes they submitted.

Scored on **term sets** — no positions, no BIO tags, no seqeval. Per note, gold
is the set of normalized highlighted phrases and prediction is the set of
normalized phrases the model returned.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

---

## ⚠️ THIS NOTEBOOK HANDLES REAL PATIENT DATA

MDACE is built on **MIMIC-III** notes — credentialed PhysioNet data.

- The sample file you upload **contains note text**. Never commit it, never
  paste note content into an issue, a chat, or a bug report.
- Run outputs go to a **mounted Drive folder** so a disconnect cannot lose the
  run. Keep that folder private.
- Only `results/*.md` and `results/*.json` are aggregate-only and safe to share.

---

## What this run costs

73 notes / **202 chunks** ≈ 2.5–3 h on a free T4. That is the union of the
50-note stratified sample and the 24 notes reachable from the 100-row sample
cut; they overlap by one note. **One inference pass produces all three scored
views** — the views differ only in which notes they count and which answer key
they use, and neither touches the model.

The run is resumable. If Colab drops, reconnect and re-run the same cell:
finished notes are skipped.

## 1. Confirm the T4 GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Install dependencies

No `seqeval` and no `datasets` here: this evaluation scores term sets from a
local JSONL, so neither the sequence scorer nor the Hub loader is used.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes tqdm huggingface_hub

## 3. Hugging Face login (gated model)

Accept the license at https://huggingface.co/google/medgemma-4b-it, then paste a
token from https://huggingface.co/settings/tokens. Read via `getpass`, so it is
never stored in the notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token (input hidden): '))

## 4. Get the project code

Code only — the repo is public and contains **no** patient data.

In [ ]:
!git clone -q https://github.com/shifat514/medgemma-ner-eval.git
%cd medgemma-ner-eval
!git checkout -q mdace-term-ner  # branch carrying the MDACE entrypoint

## 5. Mount Drive for run output

**This is what makes a 2.7 h run survivable.** Colab's own disk is wiped when the
runtime recycles; a Drive folder is not. `MDACE_OUTPUT_DIR` redirects the
per-note state there, so a disconnect costs only the note in flight.

Two files land in that folder:

| file | contents |
|---|---|
| `per_note.jsonl` | integer counts only — no note-derived text |
| `extracted_terms.jsonl` | the per-note term lists, for the term → ICD lookup |

They are split on purpose: the counts file can be opened and shared freely,
while the terms file holds phrases copied out of patient notes.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/mdace-eval-outputs'
os.makedirs(OUT, exist_ok=True)
os.environ['MDACE_OUTPUT_DIR'] = OUT
print('run output ->', OUT)
print('existing run dirs:', os.listdir(OUT) or '(none yet)')

## 6. Upload the sample file BY HAND

Build it first on the machine that holds the MDACE files:

```bash
python -m src.build_mdace_sample
```

Then run the cell below and pick `data/samples/mdace_sample.jsonl`.

The file goes to this VM's ephemeral disk, not to Drive — note text should not
outlive the runtime.

In [ ]:
import json, os, shutil
from google.colab import files

os.makedirs('data/samples', exist_ok=True)
DEST = 'data/samples/mdace_sample.jsonl'

if os.path.exists(DEST):
    print('already present:', DEST)
else:
    uploaded = files.upload()  # choose mdace_sample.jsonl
    name = next(iter(uploaded))
    if name != DEST:
        shutil.move(name, DEST)

# Sanity check WITHOUT printing note text.
recs = [json.loads(l) for l in open(DEST, encoding='utf-8')]
strat = [r for r in recs if r.get('in_stratified')]
ship = [r for r in recs if r.get('in_sample100')]
print(f'notes            {len(recs)}   (expect 73)')
print(f'  stratified     {len(strat)}   chunks {sum(r["n_chunks"] for r in strat)}'
      f'   gold terms {sum(len(r["gold_terms"]) for r in strat)}   (expect 50 / 122 / 324)')
print(f'  sample_100     {len(ship)}   chunks {sum(r["n_chunks"] for r in ship)}   (expect 24 / 82)')
print(f'chunks total     {sum(r["n_chunks"] for r in recs)}   (expect 202)')

## 7. Harness check — no GPU, ~10 seconds

Feeds the gold terms back through chunking, normalization, parsing and scoring
with no model involved. **Views B1 and B2 must read P/R/F1 = 1.0000.** Anything
less is a bug in the harness, not a model result, and there is no point burning
GPU hours until it is fixed.

View A1 will read 0.0000 — that is correct and is the point of that view: even a
*perfect* extractor scores zero against the `gold_code_description` answer key,
because the catalogue wording and the note wording are different strings.

In [ ]:
!python -m src.evaluate_mdace --limit 50

## 10b. Phase 2 — the remaining 23 notes, ~1.1 h

Adds the 80 chunks phase 1 did not run, unlocking views **A1** and **B1**: the
24 notes reachable from the 100-row sample cut, scored his way and then the
corrected way.

Phase 1's 50 notes are cached and skipped, so this costs 1.1 h, not 2.7 h.
Run it when you want the full three-view ladder for the write-up.

In [ ]:
!python -m src.evaluate_mdace

## 8. Smoke test — 5 notes, ~12 minutes

These 5 are **not** the first 5 in the file. They are chosen to span the risky
dimensions: the longest note in the sample (3,151 words / 10 chunks), plus short
single-chunk ones, covering both chart types. A head-of-file smoke test would
run five short Profee notes and never touch the path where OOM and truncation
actually happen.

15 of the 202 chunks — 7% of the work, for the thing most likely to fail.

**Watch for:**
- CUDA OOM → lower `--chunk-words` (try 300, then 250).
- `generation hit max_new_tokens` **> 0** → replies were truncated mid-JSON and
  recall is understated. Add `--max-new-tokens 1024` to every later run.
- `returned no usable JSON` high → the parser is missing a reply shape; inspect
  the raw replies in the next cell.

In [ ]:
!python -m src.evaluate_mdace --smoke 5 --dump-replies

## 9. What did the model actually say?

Reply **shapes and counts** only. The replies themselves quote note text, so the
raw text is not printed here by default — set `SHOW_ONE = True` if you need to
debug a parse failure, and clear the cell output afterwards.

In [ ]:
import glob, json, os
from collections import Counter

SHOW_ONE = False  # True prints one raw reply — CONTAINS NOTE TEXT

paths = glob.glob(os.path.join(os.environ['MDACE_OUTPUT_DIR'], '*', 'raw_replies.jsonl'))
if not paths:
    print('no raw_replies.jsonl — run the smoke cell with --dump-replies')
else:
    rows = [json.loads(l) for l in open(sorted(paths)[-1], encoding='utf-8')]
    print('chunks:', len(rows))
    print('shapes:', dict(Counter(r['shape'] for r in rows)))
    kept = [r['n_kept'] for r in rows]
    print('terms kept per chunk: min %d, median %d, max %d'
          % (min(kept), sorted(kept)[len(kept) // 2], max(kept)))
    print('chunks yielding zero terms:', sum(1 for k in kept if k == 0))
    if SHOW_ONE:
        print('\n--- one raw reply (CONTAINS NOTE TEXT) ---\n')
        print(rows[0]['reply'][:2000])

## 10. Phase 1 — the stratified 50 notes, ~1.6 h

**This is the headline result on its own.** 122 of the 202 chunks: 25 Profee +
25 Inpatient, seed 13. It produces view B2, Profee and Inpatient reported
separately, which is the number that answers "how good is MedGemma at this?"

The sample file is ordered stratified-first, so `--limit 50` is exactly that
draw — not an arbitrary first-50.

Stop here if you want. Phase 2 only adds the comparison against the 100-row
sample cut; it changes nothing about B2.

**If Colab disconnects, re-run this cell.** Finished notes are read back from
Drive and skipped.

In [ ]:
!python -m src.evaluate_mdace

## 11. Read the report

Aggregate metrics only — no note text, no term strings. Safe to share.

**Read recall as the result.** Precision on Inpatient notes is bounded by
annotation scope rather than model quality: MDACE marks evidence only for codes
that were actually billed, so a correct extraction of an unbilled condition
counts as a false positive. The report gives the best precision achievable and
splits every false positive three ways so that is visible rather than implied.

In [ ]:
import glob
from IPython.display import Markdown, display

for path in sorted(glob.glob('results/mdace_ner_*.md')):
    print('=' * 70, '\n', path, '\n', '=' * 70)
    display(Markdown(open(path, encoding='utf-8').read()))

## 12. Save the outputs

Copies the committable artifacts to Drive alongside the run state.

`extracted_terms.jsonl` is already on Drive from the run itself — that is the
file the **term → ICD lookup (experiment 2)** consumes, so the lookup can be
evaluated later without spending these GPU hours again.

In [ ]:
import glob, os, shutil

dest = os.path.join(os.environ['MDACE_OUTPUT_DIR'], 'results')
os.makedirs(dest, exist_ok=True)
for path in glob.glob('results/mdace_ner_*'):
    shutil.copy(path, dest)
    print('saved', os.path.basename(path))

print('\nrun state on Drive:')
for root, _dirs, filenames in os.walk(os.environ['MDACE_OUTPUT_DIR']):
    for name in filenames:
        full = os.path.join(root, name)
        print('  %8.1f KB  %s' % (os.path.getsize(full) / 1024,
                                  os.path.relpath(full, os.environ['MDACE_OUTPUT_DIR'])))